# LLM autoresearch: JAX + Pallas · TPU v6e / v5e

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/autoresearch_llm_tpu_v5e.ipynb)

Roofline 노트북과 같은 decoder-only Transformer의 **모델 전체 추론 시간**을 줄이는 실험이다.
모델 구조와 가중치를 유지하면서 framework 연산과 직접 만든 커널을 함께 사용할 수 있다.
`model.py`만 수정하며, 고정된 채점기가 logits와 모든 KV cache의 정확도를 검사한다.

런타임을 **TPU v6e**로 선택하고, 사용할 수 없으면 **TPU v5e**로 선택한다. prefill과 decode는 각각 별도 브랜치에서 실험한다.
이 노트북은 준비·진행 확인·결과 저장을 맡으며, 에이전트는 Colab 터미널에서 실행한다.
모델은 무작위 초기화 가중치를 사용한다. 학습이나 언어 모델 품질 개선 실험은 아니다.

## 1. 저장소와 실험 단계 선택

`PHASE`는 `prefill` 또는 `decode`로 설정한다. 진행 중인 checkout은 자동으로 pull하거나 초기화하지 않는다.
TPU는 한 프로세스만 장치를 잡도록 환경 확인과 벤치마크를 모두 별도 프로세스에서 실행한다.

In [ ]:
from pathlib import Path
import datetime
import json
import os
import subprocess
import sys

PHASE = "decode"  # "prefill" 또는 "decode"
assert PHASE in ("prefill", "decode")
TAG = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
REPO = Path("/content/tensor2silicon")
WORKDIR = REPO / "autoresearch/llm_tpu"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Jake-Song/tensor2silicon.git", str(REPO)], check=True)
assert WORKDIR.is_dir(), "새 LLM autoresearch 파일이 포함된 브랜치를 먼저 checkout하세요."
subprocess.run(["git", "-C", str(REPO), "config", "user.name", "autoresearch"], check=True)
subprocess.run(["git", "-C", str(REPO), "config", "user.email", "autoresearch@localhost"], check=True)
print("phase:", PHASE, "| tag:", TAG, "| directory:", WORKDIR)

## 2. 환경과 작은 모델 확인

이 셀은 패키지를 자동 업그레이드하지 않는다. 런타임의 라이브러리로 작은 모델의 두 단계 정확도와 선택 단계 실행을 확인한다.
GPU에는 `torch`, `triton`, TPU에는 호환되는 `jax`, `jaxlib`, `libtpu`가 필요하다. 오류가 나면 환경을 수정한 뒤 다시 실행한다.
`--smoke` 결과는 성능 기록으로 사용하지 않는다. GPU smoke도 Triton 실행을 위해 CUDA가 필요하다.

In [ ]:
def run_bench(*extra, output_name):
    output = WORKDIR / output_name
    output.unlink(missing_ok=True)  # timeout 뒤에 이전 결과를 잘못 읽지 않도록 제거
    command = [sys.executable, "bench.py", "--phase", PHASE, "--json", str(output), *extra]
    log = output.with_suffix(".log")
    try:
        with log.open("w") as f:
            completed = subprocess.run(command, cwd=WORKDIR, stdout=f, stderr=subprocess.STDOUT, timeout=900)
    except subprocess.TimeoutExpired:
        print(log.read_text()[-6000:])
        raise RuntimeError("900초 timeout: 로그를 확인하세요. 이전 측정값으로 진행하지 않습니다.")
    print(log.read_text()[-6000:])
    completed.check_returncode()
    result = json.loads(output.read_text())
    assert result["status"] == "ok" and result["correct"]
    return result

(WORKDIR / "runs").mkdir(exist_ok=True)
smoke = run_bench("--smoke", output_name="runs/smoke.json")
assert smoke["scored"] is False

## 3. 전체 모델 기준선

선택한 단계의 `colab` 크기를 측정한다. A100 또는 TPU v5e/v6e가 아니면 scored 실행은 실패한다.
처음 실행에는 컴파일 시간이 포함되므로 기다린다. `time_ms`는 컴파일·warmup을 제외한 전체 forward 시간이다.
에이전트가 실행 중일 때 이 셀을 동시에 실행하지 않는다.

In [ ]:
baseline = run_bench(output_name="baseline.json")
assert baseline["scored"] is True
print(f"Initial candidate: {baseline['time_ms']:.3f} ms | reference: {baseline['baseline_ms']:.3f} ms")
print("versions:", baseline["versions"])

## 4. 에이전트 시작

기존 autoresearch launcher처럼 Colab 터미널에서 Claude Code를 사용한다. 이미 설치되어 있으면 설치 셀은 생략한다.
인증은 터미널의 로그인 절차를 따른다. 키를 노트북이나 저장소에 기록하지 않는다.

In [ ]:
# 선택: Claude Code가 없는 런타임에서만 실행한다.
subprocess.run(["bash", "-lc", "curl -fsSL https://claude.ai/install.sh | bash"], check=True)

In [ ]:
print(f"cd {WORKDIR}")
print('export PATH="$HOME/.local/bin:$PATH"')
print("claude")
print(f"Read program.md and run full-model autoresearch. Tag: {TAG}. Phase: {PHASE}. Stop after 30 experiments.")

에이전트는 `autoresearch/<tag>-llm-tpu-<phase>` 브랜치의 독립 worktree에서 실험한다.
시작 후 아래 `RUN_DIR`을 **에이전트가 알려 준 실제 worktree의 `autoresearch/llm_tpu` 경로**로 바꾼다.
성능 향상은 기존 최고보다 1% 넘게 빠르고, 새 프로세스의 확인 실행도 같은 조건을 만족해야 인정한다.
실험별 가설·결과·교훈은 실패를 포함해 모두 `report.md`와 `results.tsv`에 남는다.

## 5. 진행 상황 보기

그래프는 선택 단계의 유효한 측정값을 보여 주며, 최고 기록은 `keep`인 실험만 반영한다.
아직 결과가 없거나 모든 시도가 실패한 경우에도 실행할 수 있다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RUN_DIR = WORKDIR  # 에이전트가 만든 worktree 안의 실제 실험 디렉터리로 변경
results_path = RUN_DIR / "results.tsv"
if not results_path.exists() or results_path.stat().st_size == 0:
    print("아직 results.tsv가 없습니다.")
else:
    df = pd.read_csv(results_path, sep="\t")
    display(df)
    for column in ("experiment", "time_ms"):
        df[column] = pd.to_numeric(df[column], errors="coerce")
    valid = df[(df.phase == PHASE) & df.status.isin(["keep", "discard"]) & (df.time_ms > 0)].copy()
    valid = valid.sort_values("experiment")
    kept = valid[valid.status == "keep"].copy()
    if valid.empty:
        print("표시할 유효한 측정값이 없습니다.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.scatter(valid.experiment, valid.time_ms, label="valid trial", color="#a4a4a4")
        if not kept.empty:
            ax.step(kept.experiment, kept.time_ms.cummin(), where="post", label="best kept", color="#2a78d6")
        ax.set(xlabel="experiment", ylabel="full forward latency (ms)", title=f"LLM autoresearch: {PHASE}")
        ax.grid(axis="y", alpha=0.25)
        ax.legend(frameon=False)
        plt.show()
report = RUN_DIR / "report.md"
if report.exists():
    display(Markdown(report.read_text()))

## 6. 결과를 Google Drive에 저장

실험 종료 후 실행한다. 실제 실험 worktree의 브랜치와 최종 모델, 로그, 보고서를 저장한다.
실험 브랜치인지 확인하며, 저장되지 않은 변경이 있으면 먼저 정리하도록 중단한다.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=RUN_DIR, text=True).strip()
assert branch.startswith("autoresearch/"), "RUN_DIR을 실제 실험 worktree 경로로 설정하세요."
tracked_changes = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=RUN_DIR, text=True)
assert not tracked_changes.strip(), "먼저 모델·보고서 변경을 커밋하세요."
out = Path("/content/drive/MyDrive/tensor2silicon-autoresearch") / branch.replace("/", "_")
out.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "bundle", "create", str(out / "branch.bundle"), branch], cwd=RUN_DIR, check=True)
for name in ("results.tsv", "report.md", "model.py", "baseline.json", "baseline.log"):
    src = RUN_DIR / name
    if src.exists():
        shutil.copy2(src, out / name)
if (RUN_DIR / "runs").exists():
    shutil.copytree(RUN_DIR / "runs", out / "runs", dirs_exist_ok=True)
print("Saved:", out)